# 篇 2 · 小文件与物理设计：读侧把 2000 个文件打包成 63 个分区，写侧的文件数有一条定律

> 要回答的坏结果：**"一张表有几百万个 KB 级文件、几十万个分区目录，`listStatus` 一次 90 秒，读它比算它还慢"**。
> 本篇不写"要合并小文件"这种口号，而是把三件事量成数：
>
> 1. **写侧**：落盘文件数怎么来的 —— 以及那条可以当场验的定律
>   `文件数 = writer(分区)数 × 每个 writer 触及的分区值数`（① 与 ⑤a）。
> 2. **读侧**：2000 个碎文件默认被打包成 **63 个输入分区**，而那个 63 是**能算出来**的（②）。
> 3. **两个旋钮各自管哪一半**：`coalesce` 与 `repartition` 的差别押在**计划里的算子名**和**落盘文件数**上，
>   不押耗时（③④）。
>
> **结论押在确定量上，耗时一律只打印。** 本篇所有 `assert` 判的都是"同一镜像每次跑都给同一个数"的那一类
> （文件数、输入分区数、每分区吃了几个文件、物理计划里的算子名、覆盖后剩下的目录数、行数守恒）。
> 读碎文件与读大文件的墙上时间（实测 0.6s vs 0.07s）**不进任何断言**，理由写在 ④。

## ⚠ 本篇顶回了一条流传很广的口径（这是它最值钱的一段）

讲小文件的资料几乎都会写："`spark.sql.files.openCostInBytes` 是虚拟打开成本，**小文件太多就是它的锅，把它调小**"。
**方向是反的。** 它是**加在每一个文件上的字节成本（加性项）**，被加进"这个文件要占多少预算"里，于是：

- 调**小** ⇒ 每个碎文件显得更便宜 ⇒ 一个分区装得下更多文件 ⇒ 输入分区数**下降**；
- 调**大** ⇒ 每个文件更贵 ⇒ 打包更保守 ⇒ 分区数**上升**，上限是"一个文件一个分区"。

本容器实测（2000 个中位 983 字节的 parquet，`maxPartitionBytes` 一直钉在出厂的 128MB）：

| `openCostInBytes` | 0 | 1MB | 2MB | **出厂 4MB** | 64MB |
| --- | --- | --- | --- | --- | --- |
| 输入分区数 | **3** | 16 | 32 | **63** | 1000 |

"2000 个文件只开出 3 个分区"这种事故，恰恰是有人**把 openCost 调小**调出来的（② 那一节当场量，
marker 也就按实测叫 `open-cost-is-additive`）。真要"每个文件一个分区"，旋钮是 `maxPartitionBytes`：
压到 1MB（openCost 保持出厂 4MB）实测恰好 **2000**。

## 本机能演示到哪一层（先说破，别拿本地的绿当分布式的证据）

- kernel 是 **`local[2]`**（一个 JVM、2 个线程、`--driver-memory 512m`，钉死在
  `docker/jupyter/kernels/arena-pyspark/kernel.json`，**不是 `local[*]`**）⇒ 本篇"写出 2000 个分区"那一步
  是**两个线程串行跑完 2000 个 task**，实测 ~20 秒。真实集群上那是一整个 stage，且失败模式完全不同。
- 没有 NameNode / 对象存储 ⇒ `listStatus` 的 90 秒、LIST/GET 请求计费、元数据把 catalog 打满，这些**成本本机测不到（未验证）**。
  本篇只给"文件数 / 目录数"这个**代理量**，不给"因此慢了多少"。
- open 成本被同一个进程的 CPU 与热页缓存掩盖 ⇒ **"更少文件 = 更快读"这句话不在本篇的断言里**：
  本机测到的收益量级（零点几秒）不足以支撑它在真实集群上的形状。
- 临时目录走 `tempfile.mkdtemp(prefix="wi90-02-")`，**不落进 `data/notebooks`**（那是读者的工作区），
  收尾 `shutil.rmtree` 并断言它真的没了；**不提交 outputs**（结果属于执行的那一刻）。

内核请选 `PySpark (arena)`（`metadata.kernelspec.name` = `arena-pyspark`）——
交付前的闸门 `server/test/notebooks/tutorials.test.ts` 会**真执行这一篇**并按内核名判：
写成 `python3` 会跑在镜像自带的系统解释器上，那正是本项目划在外面的那条隔离红线。
闸门还按 `server/test/notebooks/tutorial-claims.ts` 登记的五条结论 marker 做**双向相等**核对：
少一条 = 结论被删/改跑/marker 漂格式；多一条 = 有结论没登记（它对闸门就隐形了）。
最后一个单元是 `spark.stop()`：闸门跑完不留活内核。

*本文里出现的每个数字都是这一篇在本容器（Spark 3.5.5 / `local[2]` / 20 万行 / 2000 个碎文件）实测出来的；
从头执行一遍实测 **56 秒**（含 Spark 冷启动，其中"写出 2000 个文件"那一步就占 20.3 秒）。
跨版本行为一律标「未在本镜像验证」。*

In [ ]:
import glob
import math
import os
import shutil
import tempfile
import time

from pyspark.sql import SparkSession, functions as f

spark = SparkSession.builder.appName("wi90-02-small-files").getOrCreate()
sc = spark.sparkContext

FACTORY_KEYS = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.files.maxPartitionBytes",
    "spark.sql.files.openCostInBytes",
    "spark.sql.files.maxPartitionNum",
    "spark.sql.files.maxRecordsPerFile",
    "spark.sql.sources.partitionOverwriteMode",
]


def conf_of(key):
    '''读一个 conf 的**当前生效值**，读不到就说读不到。

    ⚠ 两个实测坑，都在这一个函数里躲掉：
    ① `SET -v` 在本镜像**不列** `spark.sql.files.openCostInBytes`（那行压根没有），但 `conf.get` 给得出 ⇒
      出厂值这一列用 `conf.get` 读，别照抄篇 1 那张 SET -v 表就说"每个键都能从 SET -v 拿到"。
    ② `spark.conf.get(key, default)` 的 default 会先被 Scala 侧按**该键的类型**转换：
      给整型键（如 `...maxPartitionNum`）传 `"(missing)"` 会抛 `IllegalArgumentException`
      而不是返回那个默认值（实测）。所以这里不传 default，改成接异常。
    '''
    try:
        return spark.conf.get(key)
    except Exception as e:  # noqa: BLE001
        return f"(读不到：{type(e).__name__})"


setv = {str(r["key"]): str(r["value"]) for r in spark.sql("SET -v").collect()}
print("出厂生效值（本镜像实测；SET -v 那一列只是对照，缺项是真的缺）")
for k in FACTORY_KEYS:
    print("   ", k, "| conf.get:", conf_of(k), "| SET -v:", setv.get(k, "(SET -v 不列这个键)"))
print("spark", spark.version, "| master", sc.master, "| defaultParallelism", sc.defaultParallelism)

# 这一行是给 ③④ 准备的**前提事实**：range 的分区数 = defaultParallelism，不是 shuffle.partitions。
df = spark.range(200_000)
N_RANGE_PARTITIONS = df.rdd.getNumPartitions()
print("spark.range(200_000) 的分区数 =", N_RANGE_PARTITIONS, "（local[2] 给的 2，不是 200，也不是 2000）")

SCRATCH = tempfile.mkdtemp(prefix="wi90-02-")
print("scratch 目录", SCRATCH, "（在容器内的 /tmp，不进 data/notebooks；⑤ 之后会被 rmtree 掉并断言没了）")

## ① 现象：你写多少个**不空的分区**，就有多少个文件

生产里小文件的来源极少是"别人塞给我的"，绝大多数是**自己的写出作业**：

- 微批作业每批一次 `df.write` ⇒ 每批都开一批新文件（批间隔越短，文件越多，见 ⑤a 那条定律）；
- 想"加速"而在写出前 `repartition(N)` ⇒ `FileFormat` 那条路**每个 writer task 写一个文件**，
  于是 `part-*.parquet` 的数量 = 提交那次写入的 task 数。

下面这一段用 `repartition(2000)` 把 2000 个碎文件**一次性做出来**（20 万行 ⇒ 每分区恰好 100 行 ⇒
实测每个文件 975–995 字节、中位 983、总计 1,966,204 字节）。
"小"要用比值说才不飘：**平均一个文件不到 1KB，而读侧默认预算是 128MB —— 差 5 个数量级**（② 就吃这个数）。

`spark.sql.shuffle.partitions` 这里一起改成 2000 只是为了让"你设的数"和"你写的分区数"是同一个数，
读者不必分辨两个 2000；`repartition(2000)` 本身是显式的，**不经过** `shuffle.partitions`（①b 会把这个差别量出来）。

⚠ 这一节把 AQE 显式关掉当**前提**，但"关掉 AQE"**并不是这个现象成立的条件**——
这是先实测才发现的：`repartition(2000)` 开着 AQE 也照样落 2000 个文件（①b 那一格当场量）。
会被 AQE 的 `coalescePartitions` 合掉的是**聚合/join 那种由 `shuffle.partitions` 推导出来的 hash shuffle**
（篇 1 实测：200 个 shuffle 分区 + AQE 开 ⇒ 只落 2 个文件）。所以配置留着当前提，
红话要说的是"配置被删了"，不是"现象消失了"。

### 这一节判什么

`文件数 == 写出时的分区数`，两个数都当场取（分区数走 `many_frame.rdd.getNumPartitions()`，
文件数走 `glob("part-*.parquet")`），并且**押的是等号**，不是押一个"总能过"的下限。
押等号的前提是"这 2000 个分区都不空"，而"空分区不产文件"这件事由 ①b 的 199 ≠ 200 单独量——
两节合起来才是完整的那句话：**文件数 = 非空 writer 数**。

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", False)
assert spark.conf.get("spark.sql.adaptive.enabled") == "false", (
    f"spark.sql.adaptive.enabled = {spark.conf.get('spark.sql.adaptive.enabled')} —— 本篇的前提是**显式控制 AQE**，"
    "红在这里说的是上面那行配置被删了/被改回去了，**不是碎文件现象消失了**："
    "实测 `repartition(2000)` 这条路开着 AQE 也仍落 2000 个文件（①b 当场量）。别改下面两条断言，把配置加回去"
)
spark.conf.set("spark.sql.shuffle.partitions", 2000)

MANY = f"{SCRATCH}/many"
many_frame = df.repartition(2000)
N_WRITE_PARTITIONS = many_frame.rdd.getNumPartitions()

t = time.time()
many_frame.write.mode("overwrite").parquet(MANY)
print("写出耗时", round(time.time() - t, 2), "s —— local[2] 上这是 2 个线程串行跑 2000 个 task 的代价，")
print("           换句话说『小文件的目录』在这台机器上是**真贵**的，别在闸门里反复跑这一段")

files = glob.glob(f"{MANY}/part-*.parquet")
sizes = sorted(os.path.getsize(p) for p in files)
entries = os.listdir(MANY)
print("文件数", len(files), "| 写出时的分区数", N_WRITE_PARTITIONS)
print("每文件字节 min/中位/max", sizes[0], sizes[len(sizes) // 2], sizes[-1], "| 总计", sum(sizes), "| 平均", round(sum(sizes) / len(files), 1))
print("目录条目", len(entries), "个，其中非数据文件（`_SUCCESS` 与 `.crc`）", len(entries) - len(files), "个 —— 数文件时别把它们算进『小文件数量』")
print("出厂读侧预算 maxPartitionBytes =", conf_of("spark.sql.files.maxPartitionBytes"), "⇒ 平均单文件与它差", round(128 * 1024 * 1024 / (sum(sizes) / len(sizes)), 1), "倍")

assert N_WRITE_PARTITIONS == 2000, N_WRITE_PARTITIONS
# 等号成立的唯一理由是"没有空分区"：round-robin 把 20 万行切成 2000 × 100 行。
# 空分区不产文件由 ①b 单独量（那里实测 200 个分区只落 199 个文件），所以这里不许把等号换成下限。
assert len(files) == N_WRITE_PARTITIONS, (len(files), N_WRITE_PARTITIONS)
assert sizes[-1] < 4096, sizes[-1]          # 实测最大 995 字节；"碎"是量出来的不是形容词
print("WI90[02-small-files-and-partitioning][file-count-is-what-you-write] OK")

## ①b 边界：AQE 合得动"它自己推导的分区数"，合不动"你钉死的那个数"

① 那句"写多少分区就有多少文件"有两个反直觉的修正，一次量完（`shuffle.partitions=200`，两种写法 × AQE 开关）：

| 写出时的形状 | AQE 关 | AQE 开 |
| --- | --- | --- |
| `df.groupBy(id % 1000).count()`：hash shuffle，分区数由 `shuffle.partitions` **推导** | **199** 个文件 | **1** 个文件 |
| `df.repartition(200)`：分区数是**用户钉死**的 | 200 个文件 | **200** 个文件 |

于是两条结论，都是本机实测而不是引用：

1. **AQE 的 `coalescePartitions` 只处理它自己推导的那一路。** `repartition(N)` 是写进计划里的显式要求，它不动
   （所以 ① 对 AQE 免疫，简报里"必须关掉 AQE 才有 2000 个文件"那句按实测被顶回）。
   生产上想少写几个文件，靠的仍然是**写出前那一步的分区数**，不是"开 AQE 它就自己合了"。
2. **199 ≠ 200 那一格说的是另一件事：空分区不产文件。** `id % 1000` 的 1000 个 key 做 hash 进 200 个桶时
   有一桶是空的，Spark 不会为它写出 `part-*.parquet`（实测那一次连 `.crc` 都没留：目录条目 400 =
   199 个数据文件 + 199 个 `.crc` + 1 个 `_SUCCESS` + 1 个 `._SUCCESS.crc`）。
   所以"文件数 = 分区数"要写成"文件数 = **非空**分区数"，而 `.crc`/`_SUCCESS` 这些元数据条目
   也不该被 `len(os.listdir(...))` 当成文件数（那正是"我数出来比你多一倍"的常见原因）。

这一节**不加 marker**：它判的还是 ① 那件事（文件数怎么来的），再加一条就是两条闸门认领同一件事。
它的 `assert` 照样被闸门的执行层判着（cell 一异常就红）。

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 200)
agg = df.groupBy((f.col("id") % 1000).alias("k")).agg(f.count(f.lit(1)).alias("n"))

counts = {}
for adaptive in (False, True):
    spark.conf.set("spark.sql.adaptive.enabled", adaptive)
    for label, frame in (("聚合 hash shuffle", agg), ("显式 repartition(200)", df.repartition(200))):
        key = ("agg" if label.startswith("聚合") else "repart", adaptive)
        p = f"{SCRATCH}/aqe_{key[0]}_{int(adaptive)}"
        t = time.time()
        frame.write.mode("overwrite").parquet(p)
        data = glob.glob(f"{p}/part-*.parquet")
        crc = [fn for fn in os.listdir(p) if fn.endswith(".crc")]
        counts[key] = len(data)
        print(f"AQE {'开' if adaptive else '关'} | {label}: 数据文件 {len(data)} 个 | 目录条目 {len(os.listdir(p))}（其中 .crc {len(crc)}）|", round(time.time() - t, 2), "s")
        shutil.rmtree(p, ignore_errors=True)

spark.conf.set("spark.sql.adaptive.enabled", False)
print("四格表", {f"{k[0]}|aqe={k[1]}": v for k, v in counts.items()})

assert counts[("agg", False)] < 200, counts             # 空分区确实存在（实测 199）
assert counts[("agg", True)] < counts[("agg", False)], counts   # AQE 确实把推导出来的那 200 合掉了
assert counts[("repart", False)] == 200, counts          # ① 那句"钉死的分区数 = 文件数"
assert counts[("repart", True)] == 200, counts           # 而 AQE 对它**无效**（这条就是顶回简报的那一句）
print("⇒ 想让 AQE 替你少写文件，只对『聚合/join 后直接写出』那一路有效；显式 repartition 它不动")

## ② 取证：读侧默认把碎文件**打包**，所以你看到的并行度从来不是文件数

碎文件躺在磁盘上，不代表读它就有 2000 个 task。`FileScan` 在**规划输入分区**时做的是贪心装箱：
每个文件的"重量"是 `min(文件字节, maxPartitionBytes) + openCostInBytes`，
装箱目标是 `maxPartitionBytes`（还会被"总重量 /(defaultParallelism × 2)"截一刀），
所以**小文件多的时候，重量里主导的那一项是 `openCostInBytes`，不是真实字节**。

① 那 2000 个文件按出厂值算：`2000 × (983 + 4,194,304) = 8,390,574,204` 字节 ÷ `128MB` = 62.5 ⇒ **63**，
实测 `getNumPartitions()` 正好给 63，且每分区吃到 16–32 个文件、`input_file_name()` 去重求和**恰好等于 2000**
（这一条很重要：它证明"合并"没漏文件，而不是只把 2000 变成了 63 却丢了 1937 个）。

三层"合并"必须分清，网上讲小文件几乎都把它们搅在一起：

| 层次 | 谁在做 | 旋钮 | 本篇哪一节量 |
| --- | --- | --- | --- |
| 读侧打包 | FileScan 规划输入分区 | `spark.sql.files.maxPartitionBytes` / `openCostInBytes` | ②（`getNumPartitions()`） |
| 写侧合并 | AQE `coalescePartitions` | `spark.sql.adaptive.*` | ①b（落盘文件数） |
| 手动合并 | `coalesce` / `repartition` | 代码本身 | ③④⑤（计划算子名 + 文件数） |

`openCostInBytes` 的**方向**就是本篇开头那条顶回，②b 那一格把 0/1/2/4/64MB 五档全量出来。
另外两个实测事实也顺手钉在这里，因为它们都会改变"分区数"这个数：

- AQE 对 `getNumPartitions()` **没有影响**（实测两档都是 63）：AQE 管的是 shuffle 之后的分区，
  不管 FileScan 的输入分区。所以 ② 的量不依赖 AQE 开关。
 - 预算压到**单个文件字节以下**时，一个 parquet 会被按字节范围切成几片：实测 512 字节预算 ⇒ **4000 个分区 = 每文件 2 片**，
  分区数**超过**文件数。⇒ "parquet 不可切分"这句老话在 Spark 3.5.5 的输入分区规划这一层不成立
  （本镜像实测；切到字节边界后**行组语义是否仍然安全：未在本镜像验证**，真要这么做应该去查社区口径而不是照抄这一段）。
- ②a 那条公式（"总重量 / 预算，向上取整"）只在**没有文件把预算顶超**的那一格对得上：
  openCost=64MB 那一格实测 1000 而公式给 1001 —— 装箱是贪心的，允许最后一个文件超出预算。
  所以那条公式是**解释**，不是判据；本篇的判据是当场取的 `getNumPartitions()`。

marker 的命名按实测走：②a 判"默认会合并"（`small-files-get-merged-on-read`），
②b 判"合并的旋钮是加性的、方向与流传口径相反"（`open-cost-is-additive`）。两条判的是不同的事，
所以值得两个 marker —— 与 ①b 的反例正好相反。

⚠ 这一节还顶掉了计划里写死的一对配置（`openCostInBytes=0` + `maxPartitionBytes=1MB`，
原预期"分区数爬回文件数量级"）：实测那一格给的是 **3**，比出厂的 63 还**少**二十倍。
按"不许把断言改成恒真"的纪律，这一节的结论与 marker 名都按实测重写了。

In [ ]:
MAX_BYTES = 128 * 1024 * 1024
OPEN_COST = 4 * 1024 * 1024
spark.conf.set("spark.sql.files.maxPartitionBytes", MAX_BYTES)
spark.conf.set("spark.sql.files.openCostInBytes", OPEN_COST)

merged = spark.read.parquet(MANY)
n_merged = merged.rdd.getNumPartitions()
print("2000 个文件读回来 ⇒ 输入分区数", n_merged, f"（比文件数少 {len(files) / n_merged:.1f} 倍）")

# "合并"要在**文件粒度**上看得见才算证明，只报分区数是不够的：input_file_name() 给每行打上它来自哪个文件
t = time.time()
files_per_partition = (
    merged.withColumn("_file", f.input_file_name())
    .rdd.mapPartitions(lambda it: iter([len({row["_file"] for row in it})]))
    .collect()
)
print("每个输入分区吃了几个文件：前 5 个分区", files_per_partition[:5], "| min", min(files_per_partition), "| max", max(files_per_partition), "| 合计", sum(files_per_partition))
print("（这一行是一遍真扫，耗时", round(time.time() - t, 2), "s —— 只打印，不进断言）")

# 63 不是魔法数：把出厂的两个旋钮代进装箱公式就能算出来（**这一档恰好落在不超出预算那一侧，所以公式对得上**；
# ②b 会给出公式失配的那一档 —— 别把这条公式当规格，它是解释不是定义）
predicted = math.ceil((sum(sizes) + OPEN_COST * len(sizes)) / MAX_BYTES)
print("按公式算：(总真实字节 + 每文件 openCost) / maxPartitionBytes =", predicted, "| 实测", n_merged)

# AQE 开关对输入分区数没有影响（实测，且这一条让 ② 的独立性可判）
spark.conf.set("spark.sql.adaptive.enabled", True)
n_merged_aqe_on = spark.read.parquet(MANY).rdd.getNumPartitions()
spark.conf.set("spark.sql.adaptive.enabled", False)
print("AQE 开同一份碎文件的输入分区数", n_merged_aqe_on, "⇒ AQE 不管 FileScan 的输入分区（本镜像实测）")

assert n_merged < len(files) // 2, (n_merged, len(files))
assert max(files_per_partition) > 1, files_per_partition          # 确实在往一个分区里塞多个文件
assert sum(files_per_partition) == len(files), (sum(files_per_partition), len(files))  # 一个都没漏
assert n_merged == predicted, (n_merged, predicted)
assert n_merged_aqe_on == n_merged, (n_merged_aqe_on, n_merged)
print("WI90[02-small-files-and-partitioning][small-files-get-merged-on-read] OK")

In [ ]:
def read_partitions(max_bytes, open_cost):
    '''按给定预算读一次 MANY，只取**输入分区数**（这是规划阶段的数，不真扫数据，所以很便宜）。'''
    spark.conf.set("spark.sql.files.maxPartitionBytes", max_bytes)
    spark.conf.set("spark.sql.files.openCostInBytes", open_cost)
    return spark.read.parquet(MANY).rdd.getNumPartitions()


MB = 1024 * 1024
n_zero = read_partitions(128 * MB, 0)
n_big = read_partitions(128 * MB, 64 * MB)
n_tight = read_partitions(1 * MB, OPEN_COST)
n_below_file = read_partitions(512, 0)
ladder = [(c, read_partitions(128 * MB, c)) for c in (1 * MB, 2 * MB, OPEN_COST)]

print(f"maxPartitionBytes 钉在 128MB，只动 openCostInBytes（{len(files)} 个中位 {sizes[len(sizes) // 2]} 字节的文件）：")
for c, n in ladder:
    print(f"    openCost={c // MB}MB → 输入分区数 {n}")
print(f"    openCost=0    → 输入分区数 {n_zero}   ← 『调小 openCost』给出来的就是这个方向")
print(f"    openCost=64MB → 输入分区数 {n_big}")
print(f"把预算压到 1MB（openCost 保持出厂 4MB）→ {n_tight}")
print(f"把预算压到 512 字节（比单个文件还小）   → {n_below_file} 个分区 / {len(files)} 个文件（一个文件被切成几片）")
# ②a 那条公式在 openCost=64MB 这一格**失配**：装箱是贪心的，允许最后一个文件把预算顶超
predicted_64 = math.ceil((sum(sizes) + 64 * MB * len(sizes)) / (128 * MB))
print(f"同一格用 ②a 那条公式算：{predicted_64}（实测 {n_big}）⇒ 公式是**解释**不是定义，别拿它当判据")

# 方向：openCost 是**加性成本**，调小 ⇒ 打包更狠 ⇒ 分区更少。与"小文件多就调小它"那句口径同向的反证。
assert n_zero < ladder[0][1], (n_zero, ladder)
assert [n for _, n in ladder] == sorted(n for _, n in ladder) and len({n for _, n in ladder}) == len(ladder), ladder
assert n_big == len(files) // 2, (n_big, len(files))   # 实测 1000 = 每分区两个文件：公式在这里失配（ceil 给 1001），因为装箱允许最后一个文件超出预算
assert n_tight == len(files), (n_tight, len(files))     # "每文件一个分区"靠的是压 maxPartitionBytes，不是动 openCost
assert n_below_file > len(files), (n_below_file, len(files))

# 恢复出厂，后面几节都跑在默认读侧预算上
spark.conf.set("spark.sql.files.maxPartitionBytes", MAX_BYTES)
spark.conf.set("spark.sql.files.openCostInBytes", OPEN_COST)
print("WI90[02-small-files-and-partitioning][open-cost-is-additive] OK")

## ③ 解法：`coalesce` 是把相邻分区**交给同一个 task**，`repartition` 才走 Exchange

判据押在**物理计划的算子名**上（同篇 1 ③B 的做法：计划文本是确定量，时间不是）。本镜像实测的两段文本：

```text
Coalesce 8                                  ← 没有 Exchange
+- *(1) ColumnarToRow
   +- FileScan parquet [id#5L] ... Location: InMemoryFileIndex(1 paths)[file:/tmp/wi90-02-.../many]

Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM     ← 全量 shuffle
+- *(1) ColumnarToRow
   +- FileScan parquet [id#5L] ...
```

| | `coalesce(8)` | `repartition(8)` |
| --- | --- | --- |
| shuffle | **无**（计划里没有 Exchange） | 有：`RoundRobinPartitioning`，带 key 时是 `hashpartitioning` |
| 能往哪个方向动 | 只能**降**到 ≤ 输入分区数 | 任意方向，且能**按 key 聚** |
| 代价 | 合并的是**相邻**分区，粒度不受控制：一个 task 可能吃下一堆不相关的数据 | 一次 shuffle = 全量落盘再读回来 |
| 什么时候用 | 数据已经在手、每个分区都不空、只想少写几个文件 | 需要按 key 聚集（`partitionBy` 写出的前奏，见 ⑤a），或需要**提高**并行度 |

取计划文本只有一条路可用（篇 1 已实测，这里照用）：`df._jdf.queryExecution().simpleString()`。
`df.explain()` 返回 `None`（它是 print）；`_jdf.executedPlan()` / `_jdf.physicalPlan()` 在本镜像 `Py4JError`；
`DataFrame.explainString` 在 3.5.5 不存在。

⚠ 还有一个在 `local[2]` 上特别容易撞的坑，③ 会把它**断**出来而不是写在正文里祈祷：
`spark.range(200_000)` 只有 2 个分区，`df.coalesce(8)` 得到的还是 **2**。
④ 里"合并写出恰好 8 个文件"那条断言必须先有一个 ≥8 分区的源（这里用的是 ② 读回来的 63 个输入分区），
否则红出来的是并行度不够，不是合并逻辑错。
顺带一条同族的坑（篇 1 实测）：**物理计划是 lazy val**，改完配置再对**同一个 DataFrame 对象**取计划，
文本一字不变 —— 所以每个要看"换配置后的计划"的地方都得**新建 DataFrame**。

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.shuffle.partitions", 200)


def plan_of(frame):
    '''物理计划文本。**每次新建 DataFrame**：计划是 lazy val，复用同一个对象时改了配置也不会重新选算子。'''
    return frame._jdf.queryExecution().simpleString()


scan = spark.read.parquet(MANY)          # ②b 结束时已恢复出厂读侧预算
assert scan.rdd.getNumPartitions() == n_merged, (scan.rdd.getNumPartitions(), n_merged)  # 配置漂了就红在这里

plan_coalesce = plan_of(scan.coalesce(8))
plan_repartition = plan_of(scan.repartition(8))
plan_by_key = plan_of(scan.repartition(8, "id"))
for name, txt in (("coalesce(8)", plan_coalesce), ("repartition(8)", plan_repartition), ("repartition(8, 'id')", plan_by_key)):
    print("---", name, "---")
    print("\n".join(ln for ln in txt.splitlines() if ln.strip() and not ln.startswith("==")))

assert "Exchange" not in plan_coalesce, plan_coalesce          # coalesce 不引入 shuffle
assert "Coalesce 8" in plan_coalesce, plan_coalesce
assert "Exchange RoundRobinPartitioning(8)" in plan_repartition, plan_repartition
assert "hashpartitioning(id" in plan_by_key, plan_by_key       # 带 key 的 repartition 才是治理碎文件的那个（⑤a）

# coalesce 只能降、不能升（④ 那条 ==8 的前提就是这里）
assert scan.coalesce(8).rdd.getNumPartitions() == 8, n_merged
assert scan.repartition(8).rdd.getNumPartitions() == 8
assert df.coalesce(8).rdd.getNumPartitions() == N_RANGE_PARTITIONS == 2, (
    f"spark.range 在 local[2] 上给的是 {N_RANGE_PARTITIONS} 个分区 ⇒ coalesce(8) 升不上去。"
    "红在这里说的是 defaultParallelism 变了（kernel.json 那个 local[2] 被动过），不是这条结论错了"
)
print("coalesce(8)：作用在", n_merged, "个输入分区上 → 8 个；作用在 spark.range 的", N_RANGE_PARTITIONS, "个分区上 →", df.coalesce(8).rdd.getNumPartitions(), "个（升不上去）")
print("WI90[02-small-files-and-partitioning][coalesce-not-shuffle] OK")

## ④ 量差异：文件数、总字节、行数守恒 —— 耗时**只打印**

同一批数据（20 万行）三种写法落盘，实测（左列是碎文件那一格，右两列是本节写的）：

| 写法 | 落盘文件数 | 总字节 | 平均单文件 |
| --- | --- | --- | --- |
| ① 的碎文件（`repartition(2000)`） | 2000 | 1,966,204 | 983 字节 |
| 读回来**原样**写（63 个输入分区） | 63 | 978,615 | 15,533 字节 |
| `coalesce(8)` 写 | **8** | 950,544 | 118,818 字节 |

断言押四件当场可判的事：**文件数正好 8**、**行数一字不变**（两种读法都 200,000）、
**合并后总字节不增**、**平均单文件涨 100 倍以上**。
总字节能降是因为 parquet 每个文件都有一份固定开销（footer + 块头 + 元数据），
2000 份 → 8 份就是本篇那 ~52% 的来源；真实集群上这份开销还包括对象存储的一次 GET/LIST，
而目标单文件是 128–256MB（`bd-pyspark-0005` 的口径），本篇的数据量到不了那个尺寸。

⚠ 上面那两个"总字节"（978,615 / 950,544）**不是逐次稳定的**：本篇另一轮实测给的是 978,523 / 950,484，
漂了约 0.01%（同一个镜像、同一份数据，差在 parquet 的字典/块编码落在哪些行上）。
碎文件那一格（2000 × ~983 字节）反倒每次一字不差。
⇒ 所以断言里只有**不等式**（`bytes_few < bytes_many`、`avg_few > 100 × avg_many`）与**计数**（8 个文件、20 万行），
没有"总字节 == 某个具体数"。正文里的这两个数读作"量级 + 形状"，别当校验和用。

**读回耗时（实测 many 0.67s / few 0.08s）只打印，不进任何断言，也不写进 marker 的名字。** 两条理由：

1. `local[2]` 是同一个 JVM 的两个线程：碎文件的"打开成本"在这里没有网络往返、没有 NameNode RPC、
   页缓存还是热的 ⇒ 本机省下的这半秒是本机噪声级别的收益，换个集群形状就变（未验证）。
2. 一条押耗时的 `assert` 就是一条每次跑都可能翻转的红。仓库的纪律是"耗时一律只打印"
   （篇 1 的 ④ 同一件事；本篇旧版计划里那句 `assert t_read_few <= t_read_many * 2` 就是按这条删掉的）。

所以这一节的结论叫 `coalesce-cuts-files-not-rows`：**它宣称的是"砍文件不砍行"，而这三件事全都有确定量在判**。
不叫 `fewer-files-not-slower-read` —— 那个名字宣称的东西没有任何确定量守着。

In [ ]:
FEW = f"{SCRATCH}/few"
AS_IS = f"{SCRATCH}/as_is"

t = time.time()
scan.coalesce(8).write.mode("overwrite").parquet(FEW)
t_few_write = round(time.time() - t, 2)
few_files = glob.glob(f"{FEW}/part-*.parquet")

t = time.time()
scan.write.mode("overwrite").parquet(AS_IS)          # 不 coalesce：文件数跟着**输入分区数**走
t_as_is_write = round(time.time() - t, 2)
as_is_files = glob.glob(f"{AS_IS}/part-*.parquet")

t = time.time()
rows_many = spark.read.parquet(MANY).count()
t_read_many = round(time.time() - t, 2)
t = time.time()
rows_few = spark.read.parquet(FEW).count()
t_read_few = round(time.time() - t, 2)

bytes_many = sum(os.path.getsize(p) for p in files)
bytes_few = sum(os.path.getsize(p) for p in few_files)
bytes_as_is = sum(os.path.getsize(p) for p in as_is_files)
avg_many, avg_few = bytes_many // len(files), bytes_few // len(few_files)

print(f"many   : {len(files):>5} 个文件 / {bytes_many:>9} 字节 / 平均 {avg_many:>6} 字节")
print(f"as_is  : {len(as_is_files):>5} 个文件 / {bytes_as_is:>9} 字节 / 平均 {bytes_as_is // len(as_is_files):>6} 字节（写侧：读回来原样写回去，{t_as_is_write} s）")
print(f"few    : {len(few_files):>5} 个文件 / {bytes_few:>9} 字节 / 平均 {avg_few:>6} 字节（coalesce(8)，{t_few_write} s）")
print(f"读回耗时（只打印）：{len(files)} 个文件 {t_read_many} s | {len(few_files)} 个文件 {t_read_few} s")
print(f"行数：many {rows_many} | few {rows_few}")

assert len(few_files) == 8, len(few_files)
assert len(as_is_files) == n_merged, (len(as_is_files), n_merged)   # 原样写 = 每个输入分区一个文件（① 那句话的又一格）
assert rows_many == rows_few == 200_000, (rows_many, rows_few)
assert bytes_few < bytes_many, (bytes_few, bytes_many)
assert avg_few > 100 * avg_many, (avg_few, avg_many)                 # 实测 118,810 vs 983 ≈ 120 倍
print("WI90[02-small-files-and-partitioning][coalesce-cuts-files-not-rows] OK")

## ⑤ 物理设计：文件数有一条定律，`partitionOverwriteMode` 决定覆盖写砸多大一块

### 5.0 定律（⑤a 当场量）

用 `partitionBy(...)` 写出时：

> **文件数 = writer(分区)数 × 每个 writer 实际触及的分区值数。**

机制只有一句：一个 task 只有一批 writer 文件句柄，但它**每碰到一个新的分区值就新开一个文件**。
所以"我明明 coalesce(8) 了，为什么落盘 240 个文件"不是 bug，是这条定律。
⑤a 用 `dt = id % 30`（30 个分区值）把四个写法量成一张表：

| 写法 | writer 数 | 每个 writer 触及的 dt 值 | 落盘文件数 | 实测写耗时 |
| --- | --- | --- | --- | --- |
| 原样写（63 个输入分区） | 63 | 30 | **1890** = 63×30 | 7.1 s |
| `coalesce(8)` | 8 | 30 | **240** = 8×30 | 1.3 s |
| `repartition(8, 'dt')` | 8 | 每值只归一个 writer | **30** | 0.8 s |
| `coalesce(1)` | 1 | 30 | **30** | 0.9 s |

第三行与第四行文件数一样，**代价完全不同**：`coalesce(1)` 是把全部数据压给一个 task
（无并行度、尾部单文件巨大、失败就整批重来），这正是 **`bd-pyspark-0005` 明令禁止的写法**；
`repartition(N, 分区键)` 是让同一个键的行聚到同一个 writer —— 文件数降到"分区值数"这个下限，
而并行度还是 N。**这才是治理"分区写出小文件"的那个动作。**

"目录爆炸"就是这么乘出来的：微批作业每批 `writer 数 × 分区值数` 个新文件，
分区值 = 日期粒度 × 租户基数，于是 `bd-pyspark-0005` 那道题的生产数字是
**41 万个分区目录、`listStatus` 一次 90 秒**（那两个数字属于题目场景，本机未验证；本机验证的是**乘法本身**）。

### 5.0b 覆盖写：`mode("overwrite")` 砸的是一张表还是一批分区

`spark.sql.sources.partitionOverwriteMode` 决定它，本镜像出厂实测是 `static`
（注意 `conf.get` 回来是大写 `'STATIC'`，比对要先 `.lower()`）。⑤b 的探针：基线 30 个 `dt=` 目录，然后只重写 `dt < 05`：

| 模式 | 目录数 之后 | 全表行数 之后 | 含义 |
| --- | --- | --- | --- |
| `static`（出厂） | 30 → **5** | **33,335** | 整张表被清掉，只留下这次写出的那些分区 ⇒ **一次清库事故** |
| `dynamic` | 30 → **30** | **200,000** | 只覆盖本次出现的分区，其余 25 个分区原样保留 |

所以"按天增量重算"的口径是 **`dynamic` + 按分区覆盖**；`static` 配 `overwrite` 写分区表就是事故
（`bd-pyspark-0008` 那道"回溯不许污染已发布分区"里，覆盖范围是它的两个根因之一）。

一次合并写的**取舍**也要说清，别把 `dynamic` 当万能：

- `dynamic` 只保证"出现的分区被整体替换"，**不保证**"这次写出就是该分区的全集"之外的任何事；
  上游少给一天就少改一天，静默漏数要靠"分区就绪"检查（`bd-pyspark-0005` 的 `min/max created` 自检就是干这个的）。
- 删除事件不在 overwrite 的语义里 ⇒ 要么 `MERGE INTO` / 主键表 upsert，要么按分区全量重算再覆盖。
- 攒批 vs 合并写是另一组取舍：批间隔决定文件数（定律里那个乘数），compaction 是把乘回来的代价延后支付
  （Iceberg `rewrite_data_files` 定时 bin-pack；湖仓口径见 `bd-pyspark-0005` 的追问）。

⑤a/⑤b **不加 marker**：它们判的还是 ① 与 ③ 那两件事的延伸（文件数怎么来的、按 key 聚分区有没有用），
两条 marker 判一件事是本仓库明确拒收的形状。它们的 `assert` 由闸门的执行层判。

In [ ]:
DT_VALUES = 30
part_frame = scan.withColumn("dt", f.lpad((f.col("id") % DT_VALUES).cast("string"), 2, "0"))


def write_partitioned(label, frame):
    '''写一次并按**文件粒度**记账：定律判的是 `文件数 == writer 数 × 分区值数`。'''
    p = f"{SCRATCH}/law_{label}"
    writers = frame.rdd.getNumPartitions()
    t = time.time()
    frame.write.mode("overwrite").partitionBy("dt").parquet(p)
    secs = round(time.time() - t, 2)
    fs = [os.path.join(r, fn) for r, _, fns in os.walk(p) for fn in fns if fn.endswith(".parquet") and not fn.startswith(".")]
    dirs = sorted(d for d in os.listdir(p) if d.startswith("dt="))
    print(f"{label:<22} writer(分区)数 {writers:>3} × dt 目录数 {len(dirs):>2} = 落盘文件数 {len(fs):>4} | {secs} s")
    return p, writers, len(dirs), len(fs)


res = {}
for label, frame in (("asis", part_frame), ("coalesce8", part_frame.coalesce(8)), ("repartition8_key", part_frame.repartition(8, "dt")), ("coalesce1", part_frame.coalesce(1))):
    res[label] = write_partitioned(label, frame)
for p, *_ in res.values():
    shutil.rmtree(p, ignore_errors=True)

# 前三格：没有按 key 聚集时，每个 writer 都会触及全部 30 个 dt ⇒ 等号成立
for label in ("asis", "coalesce8", "coalesce1"):
    writers, values, nfiles = res[label][1:]
    assert nfiles == writers * values, (label, writers, values, nfiles)
# 第四格是按 key 聚集的那一个：8 个 writer 却只有 30 个文件 ⇒ 定律里的第二个因子变成了 1
w_key, v_key, f_key = res["repartition8_key"][1:]
assert w_key == 8 and f_key == DT_VALUES == v_key, res["repartition8_key"]
assert res["coalesce8"][3] > f_key, res               # 同样 8 个 writer：不聚 key 就是 240 vs 30
assert res["asis"][3] == n_merged * DT_VALUES, res     # 63 × 30 = 1890：读一遍再分区写一遍，文件数就乘回去了
print(f"⇒ 『coalesce(8) 治小文件』在分区表上是假的：{res['coalesce8'][3]} 个文件；按 key 聚之后同样 8 个 writer 只给 {f_key} 个")

In [ ]:
probe = {}
base = f"{SCRATCH}/overwrite_probe"
recent = part_frame.filter(f.col("dt") < "05")          # 只涉及 dt=00..04 这 5 个分区值

for mode in ("static", "dynamic"):
    # ⚠ 每种模式都要**重新灌一遍基线**：上一轮 static 的覆盖已经把基线砍成 5 个目录了，
    # 拿它接着验 dynamic 会得到"dynamic 也只留 5 个"这种假结论 —— 本篇的探针第一版就是这样测错的（实测）。
    shutil.rmtree(base, ignore_errors=True)
    part_frame.coalesce(4).write.mode("overwrite").partitionBy("dt").parquet(base)
    before = sorted(d for d in os.listdir(base) if d.startswith("dt="))
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", mode)
    t = time.time()
    recent.coalesce(2).write.mode("overwrite").partitionBy("dt").parquet(base)
    after = sorted(d for d in os.listdir(base) if d.startswith("dt="))
    rows = spark.read.parquet(base).count()
    probe[mode] = (len(before), len(after), rows)
    print(f"partitionOverwriteMode={mode:<8} 目录 {len(before)} → {len(after)} | 全表行数 → {rows} | 覆盖写耗时 {round(time.time() - t, 2)} s（只打印）")
    print(f"    被删掉的目录：{[d for d in before if d not in after][:3]}{' ...' if len([d for d in before if d not in after]) > 3 else ''}")

shutil.rmtree(base, ignore_errors=True)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")   # 恢复出厂（本镜像出厂就是 static）
assert conf_of("spark.sql.sources.partitionOverwriteMode").lower() == "static", conf_of("spark.sql.sources.partitionOverwriteMode")

assert probe["static"][1] == 5, probe            # 出厂模式：只写出 5 个分区，整张表就剩 5 个目录
assert probe["dynamic"][1] == probe["dynamic"][0] == DT_VALUES, probe
assert probe["static"][2] < probe["dynamic"][2] == 200_000, probe
print(f"⇒ 出厂的 static 配 mode(overwrite) 写分区表 = 一次清库（{probe['static'][0]} 个目录 → {probe['static'][1]} 个）；增量重算要 dynamic")

## ⑤ 速查表（可以只靠它回答问题）+ 追问链

### 5.1 线上发现"文件/目录太多"，先查哪四个数（都不需要时间）

| # | 查什么 | 怎么查 | 判什么 |
| --- | --- | --- | --- |
| 1 | **每个分区的文件数分布**（不是总数） | `hdfs dfs -count` / 对象存储清单；本机：本篇 ⑤a 的 `os.walk` + `dt=` 目录数 | 总数 = writer 数 × 分区值数 ⇒ 是**写侧并行度 × 分区基数**乘出来的，不是"上游给的碎" |
| 2 | **单文件字节分布**（min/中位/max） | 本篇 ①（`os.path.getsize`） | 中位 ≪ `maxPartitionBytes`（出厂 128MB）⇒ 每个文件都远没填满预算；中位 ≈ 预算 ⇒ 不是小文件问题，是**分区数问题** |
| 3 | **读它时开几个输入分区** | `spark.read.parquet(p).rdd.getNumPartitions()` | 远小于文件数 ⇒ 读侧已经在打包，"task 太多"这个诊断是错的（真正贵的是元数据与 open，见 4） |
| 4 | **每分区吃了哪些文件、有没有漏** | 本篇 ②a 的 `input_file_name()` 去重求和 | 求和 == 文件数 ⇒ 合并没漏文件；不等 ⇒ 目录里有并发写/半途失败的残留（要看 `_spark_metadata`/`.crc`） |

再往下分叉只看一件事：**这些文件是"读侧碍事"还是"写侧在继续长"**。
读侧碍事 → 调 `maxPartitionBytes`（②b 那张表）；写侧还在长 → 动写出前的分区数与分区键（③⑤a），
**别去调 `openCostInBytes` 治写侧**：它对已经躺在磁盘上的文件生效，对"为什么会写出这么多文件"无能为力。

### 5.2 参数表（默认值那一列是本镜像实测，`conf.get` 读，不是背的）

| 参数 | 本镜像生效值 | 什么时候动 | 动错的表征 |
| --- | --- | --- | --- |
| `spark.sql.files.maxPartitionBytes` | `128MB` | 读侧装箱预算。碎文件想多开并行度 → 调**小**它（实测 1MB ⇒ 一文件一分区）；大文件想少开 → 调大 | 调到单文件字节以下 ⇒ 一个 parquet 被切成几片（实测 512B ⇒ 4000 个分区 > 2000 个文件）；切分后的行组语义**未在本镜像验证** |
| `spark.sql.files.openCostInBytes` | `4194304b`（4MB）；⚠ `SET -v` 不列这个键，只能用 `conf.get` 读 | 它是**加在每个文件上的字节成本**：调大 ⇒ 分区更多，调小 ⇒ 打包更狠 | **方向被流传口径讲反**：设 0 ⇒ 2000 个碎文件只开 **3** 个分区（实测），读它的那个 job 变成 3 个 task 串行啃 2000 个文件 |
| `spark.sql.files.maxPartitionNum` | 未设（`None` ⇒ 不封顶） | 想给读侧分区数一个硬上限时（例如下游并发被压垮） | 设太小 ⇒ 单个输入分区吃掉海量文件，task 时长与 spill 失控 |
| `spark.sql.files.maxRecordsPerFile` | `0`（不限）；⚠ 网上大量资料写的旧键名 `spark.sql.maxRecordsPerFile` 在本镜像 `conf.get` **读不到** | 给单文件封顶行数（写侧防"巨大单文件"） | 配 `coalesce(1)` 时行数上限会把它拆成多个文件，"文件数=writer 数"这条定律要改读成"× 每 writer 的文件数" |
| `spark.sql.sources.partitionOverwriteMode` | `'STATIC'`（大写，比对要 `.lower()`） | 增量/按天重算一律 `dynamic` | `static` + `overwrite` 写分区表 = 清库（⑤b 实测 30 个目录 → 5） |
| `spark.sql.adaptive.enabled` | `true`（本篇显式关掉当度量前提） | 度量/教学时关；生产开 | 以为它会替你治碎文件：它只合**它自己推导的** shuffle 分区，`repartition(N)` 与 `FileScan` 都不归它管（①b、②a 各量一次） |
| `spark.sql.adaptive.coalescePartitions.minPartitionSize` | `1MB` | 阻止过度合并（把小分区留给下游并行度） | 调大 ⇒ 并行度被压没；调小 ⇒ 上万个碎分区（这是**读 shuffle 之后**的那一层，不是读侧装箱） |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | `67108864b`（64MB；`SET -v` 里它显示成别名 `<value of spark.sql.adaptive.shuffle.targetPostShuffleInputSize>`） | join/shuffle 侧目标分区大小 | 与上面那个 minPartitionSize 混为一谈 → "为什么合并动不动"。**Spark 4.0 改名 `advisoryPartitionSizeThreshold`：未在本镜像验证** |
| `spark.sql.shuffle.partitions` | `200` | 只决定"shuffle 之后开多少个分区/写多少个文件"，**不解决热点 key**（见篇 1 的 C 组） | 调太大 ⇒ 一次写出几千个 KB 文件（本篇 ① 就是这个形状）；调太小 ⇒ 单 task 太大、spill |
| `local[2]` 的 `defaultParallelism` | `2` | 本机一切"并行度"数字的分母 | 看到"200 个 shuffle 分区只落 2 个文件"（篇 1）或"读 1.97MB 只开 3 个分区"（本篇 ②b）别慌：那都是 `total/(defaultParallelism×2)` 那一刀 |

**治理动作的形状（背这一句就够）**：
`读侧不匹配 → maxPartitionBytes/openCostInBytes`；
`写侧文件多 → 写出前的分区数（coalesce 降 / repartition 升）`；
`分区表写侧文件多 → repartition(N, 分区键) 而不是 coalesce(1)`；
`增量覆盖 → partitionOverwriteMode=dynamic`；
`历史存量 → 一次合并写（读回来按 key 聚再写）或表级 compaction（Iceberg rewrite_data_files / Paimon full compaction）`。

### 5.3 面试官追问链（第三条是机制层，本篇的落点）

1. **"你怎么判断这是小文件问题而不是数据量大？"**
   数**分布**不数总量：单文件中位数 vs `maxPartitionBytes`（实测 983 字节 vs 128MB ⇒ 差 5 个数量级），
   再看每分区文件数（`input_file_name()` 去重，实测 16–32 个/分区）。
   追问：**"那目录爆炸和文件爆炸是同一件事吗？"** —— 不是。目录数由**分区值的基数**决定（日期粒度 × 租户），
   文件数 = 目录数 × writer 数（⑤a 的 1890 = 63×30）；治理动作也不同：前者改分区键粒度，后者改写出并行度。
2. **"读一堆小文件为什么会慢？慢在哪？"**
   三个地方：每个文件的 open/元数据往返（对象存储按请求计费）、footer 解析、task 调度与启动开销；
   而 Spark 已经在读侧打包（实测 2000 → 63），所以"task 数量 = 文件数量"这句是错的。
   追问：**"那 `openCostInBytes` 调小是不是就不过度合并了？"** —— 反了（本篇开头那一格表：0 ⇒ 3 个分区）。
   它是加性成本：想让碎文件多开并行度，要么调**大**它，要么调**小** `maxPartitionBytes`。
3. **"为什么 `coalesce` 不能随便用在分区表上？"**（机制层）
   `coalesce` 只把**相邻**的输入分区交给同一个 task，它既不知道也不关心分区值：
   于是一个 task 会依次打开它遇到的每个分区值的 writer ⇒ 文件数 = writer 数 × 分区值数（实测 240 = 8×30）。
   要让文件数落到"分区值数"这个下限，必须让同一个分区值的行**聚到同一个 task**，
   那是 `repartition(N, 分区键)` 的 `hashpartitioning(键, N)`（⑤a 实测同一 writer 数下 30 个文件）。
   追问：**"那 `coalesce(1)` 也能得到 30 个文件，为什么它是被禁止的写法？"** —— 文件数一样，
   **并行度**不一样：⑤a 里 `coalesce(1)` 与 `repartition(8,'dt')` 都是 30 个文件，
   前者是 1 个 task 扛全部数据（无并行、尾部长、失败整批重来），后者是 8 个 task 各扛一段。
   "文件数少"不是唯一的成本项，这就是为什么这条要实测而不是嘴上说。
4. **"增量重算怎么写才不会把历史清掉？"**
   `partitionOverwriteMode=dynamic` + 按分区覆盖（⑤b 实测 static 会把 30 个目录砍成 5 个）；
   并且要知道 `dynamic` 的边界：它只替换"这次出现"的分区 ⇒ 上游少给一天就静默少改一天，
   所以要有分区就绪自检（`bd-pyspark-0005` 的 `min/max created` 那类不变量），删除事件要另走 MERGE/CDC。
   追问：**"湖仓里呢？"** —— `MERGE INTO`/主键表 upsert，compaction 做成表的一等公民（口径见 `bd-pyspark-0005`，
   Iceberg/Paimon 的具体参数**未在本镜像验证**，这里没有湖仓表）。

### 5.4 本篇没验的东西（照纪律 2 逐条列出）

- **元数据与请求成本**：`listStatus` 90 秒、对象存储 LIST/GET 计费、NameNode/catalog 打满 —— 本机没有这些层，
  只给"文件数/目录数"这个代理量。
- **"更少文件 = 更快读"的真实量级**：本机 0.60s vs 0.07s 不构成结论（同进程、热缓存、无网络往返）。
- 湖仓表格式（Iceberg `rewrite_data_files`、Paimon `num-sorted-run.stop-trigger` 与 full compaction、`MERGE INTO`）
  —— 本镜像没有这些 catalog，本篇只在口径层引用 `bd-pyspark-0005`。
- `maxPartitionBytes` 压到单文件以下时**按字节切分 parquet**的行组语义安全性：只实测了"会切成 4000 片"这个事实。
- Spark 4.x 的键名漂移（`advisoryPartitionSizeThreshold`、旧 `spark.sql.maxRecordsPerFile` 是否有别名兼容）：本镜像只有 3.5.5。
- `dynamic` 与并发写/流式 source 的交互（`_spark_metadata`、`maxFilesPerTrigger` 那一族）：本篇是批写批读，未涉及。

## 题库指针（同一套口径，不是"再找篇资料"）

- **`bd-pyspark-0005`** ——「分区键治理 + at-least-once 重复投递：目录爆炸与 `__HIVE_DEFAULT_PARTITION__` 的根治」（senior，判题 `pyspark`）。
  这就是本篇面试题的本体，而且它的**量法**正是 ⑤a 那条定律：生产场景给的是
  日均 4.7 亿行、900 天、有人拿 `ingest_ts` 的**秒**做分区列 ⇒ **41 万个分区目录**、`listStatus` 90 秒。
  题面里有两条红线与本篇直接对上，写法上别踩：
  ① **禁止 `coalesce(1)` 治小文件**（⑤a 实测了它"文件数确实最少但只有一个 task"）；
  ② 正确姿势是写出前 `repartition('event_date','tenant_bucket')`（⑤a 的第四格，实测同样 writer 数下 240 → 30）。
  它的参考要点还有两条本篇量不到的（目标单文件 128–256MB、微批提交间隔决定文件数），
  以及本篇 ⑤b 只碰了一半的那条：`partitionOverwriteMode=dynamic` + 幂等去重（本题还要处理重复投递的**归属**，
  被丢弃的重复行要计入**保留行所在分区** —— 那是数据正确性，不是文件布局）。
- **`bd-pyspark-0008`** ——「回溯不许污染已发布分区：冻结日拦截 + 未发布日重算」。
  对应本篇 ⑤b 的覆盖面那一半：重算的单位是**分区**，而覆盖模式选错（`static`）就是清库事故；
  它还多一层本篇没有的东西 —— 分区"能不能写"取决于**是否已发布**这个外部事实，不取决于数据新旧。
- **`bd-pyspark-0003`** ——「版本化 upsert」。当"按分区全量覆盖"太粗、又要处理迟到与删除时，
  解就落到 upsert/MERGE 那一侧（⑤b 末尾那条边界）。

系列内：篇 1 = 倾斜与热点 key（③ 的"分区数不解决热点 key"、①b 引用的"AQE 把 200 个 shuffle 分区合成 2 个文件"都在那篇实测）；
篇 3 = 计划解读与 AQE 三段能力。本篇刻意**不** `cache()`：一 cache，后面几次读侧打包与文件数就量的不是那批碎文件本身了。

In [ ]:
# 收尾断言必须留在**数据单元之后、spark.stop() 之前**：闸门每跑一次这一篇，
# 就要在容器里少留一份 2000 个 parquet 的垃圾。破坏性验证（删掉 rmtree）红出来的是这一条。
shutil.rmtree(SCRATCH, ignore_errors=True)
assert not os.path.exists(SCRATCH), (
    f"scratch 没清干净：{SCRATCH} 还在 ⇒ 闸门每跑一次就在容器 /tmp 里留 2000 个碎文件。"
    "红在这里说的是上面那行 rmtree 被删了/失败了，不是'多点垃圾无所谓'"
)
leftover = [d for d in os.listdir(tempfile.gettempdir()) if d.startswith("wi90-02-")]
print("清理完成 | /tmp 里以 wi90-02- 开头的残留：", leftover)
assert not leftover, leftover
print("本篇耗时（不含 Spark 冷启动，只到这一行为止的各步之和）见各单元打印")

In [ ]:
# 最后一个单元必须是它：闸门（tutorials.test.ts）跑完这一篇就结束，
# 不往容器里丢活着的 Spark 内核 —— 详见 WI-93 那条"没有守护"的教训的反面。
spark.stop()
print("stopped")